# 07 - IOHanalyzer Benchmarking & Comparison

This notebook:
1. Loads performance trajectory logs (`.dat` files) from `data/ioh_logs/` for target BBOB problems (`f1, f8, f11, f15, f21`) at `dim=5`, `noise_std=0.05`.
2. Compares **LLaMEA Champion** against classical baselines (**CMA-ES**, **DE**, **PSO**).
3. Computes:
   - **Fixed-Budget Convergence Curves** (Mean ground-truth clean error vs. Function Evaluations).
   - **Empirical Cumulative Distribution Functions (ECDF)** over target error thresholds.
4. Exports thesis-ready PDF/PNG figures to `writing/thesis/figures/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling for thesis
plt.style.use('seaborn-v0_8-paper' if 'seaborn-v0_8-paper' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 300
})

PROJECT_ROOT = Path('../').resolve()
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'
FIGURES_DIR = PROJECT_ROOT / 'writing' / 'thesis' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PROBLEMS = [1, 8, 11, 15, 21]
ALGO_COLORS = {
    'LLaMEA Champion': '#d62728',  # Red
    'CMA-ES': '#1f77b4',           # Blue
    'DE': '#2ca02c',               # Green
    'PSO': '#ff7f0e'               # Orange
}

print(f'IOH Logs Path: {IOH_LOGS_DIR}')
print(f'Figures Export Path: {FIGURES_DIR}')

## 1. Load IOH Trajectory Data

In [ ]:
def load_ioh_dat_files(problem_dir: Path):
    """Load all .dat files grouped by algorithm from a problem directory."""
    data = {}
    if not problem_dir.exists():
        return data
        
    for algo_dir in problem_dir.iterdir():
        if not algo_dir.is_dir():
            continue
        algo_name = algo_dir.name
        if algo_name == 'llamea_champion':
            clean_name = 'LLaMEA Champion'
        else:
            clean_name = algo_name.upper()
            
        runs = []
        for dat_file in algo_dir.rglob('*.dat'):
            try:
                df = pd.read_csv(dat_file, sep=r'\s+', comment='#')
                if len(df) > 0 and 'evaluations' in df.columns and 'raw_y' in df.columns:
                    runs.append(df)
            except Exception as e:
                print(f'Failed to parse {dat_file}: {e}')
                
        if runs:
            data[clean_name] = runs
    return data

print('Data loading helper defined.')

## 2. Fixed-Budget Convergence & ECDF Plots

In [ ]:
for p_id in TARGET_PROBLEMS:
    prob_dir = IOH_LOGS_DIR / f'f{p_id}_5D_std0.05'
    algo_runs = load_ioh_dat_files(prob_dir)
    
    if not algo_runs:
        print(f'f{p_id}: No IOH log files found at {prob_dir}. Run Notebooks 05 and 06 first.')
        continue
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Subplot 1: Convergence (Clean Error vs Evaluations) ---
    for algo_name, runs in algo_runs.items():
        color = ALGO_COLORS.get(algo_name, '#7f7f7f')
        eval_grid = np.logspace(0, 5, 200)
        interp_y = []
        for df in runs:
            evals = df['evaluations'].values
            errors = df['raw_y'].values
            cum_min_errors = np.minimum.accumulate(errors)
            y_interp = np.interp(eval_grid, evals, cum_min_errors, left=cum_min_errors[0], right=cum_min_errors[-1])
            interp_y.append(y_interp)
            
        mean_y = np.mean(interp_y, axis=0)
        std_y = np.std(interp_y, axis=0)
        
        ax1.plot(eval_grid, mean_y, label=algo_name, color=color, linewidth=2)
        ax1.fill_between(eval_grid, np.maximum(mean_y - std_y, 1e-12), mean_y + std_y, color=color, alpha=0.15)
        
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('Function Evaluations')
    ax1.set_ylabel('Clean Distance to Optimum $\\Delta f(x)$')
    ax1.set_title(f'BBOB f{p_id} (5D, Noise std=0.05) - Convergence')
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.legend()
    
    # --- Subplot 2: ECDF over target thresholds ---
    targets = np.logspace(-8, 2, 100)
    for algo_name, runs in algo_runs.items():
        color = ALGO_COLORS.get(algo_name, '#7f7f7f')
        hit_rates = []
        for t in targets:
            hits = sum(1 for df in runs if np.min(df['raw_y'].values) <= t)
            hit_rates.append(hits / len(runs))
            
        ax2.plot(targets, hit_rates, label=algo_name, color=color, linewidth=2)
        
    ax2.set_xscale('log')
    ax2.set_xlabel('Target Precision $\\tau$')
    ax2.set_ylabel('ECDF (Fraction of Successful Trials)')
    ax2.set_title(f'BBOB f{p_id} - ECDF Curve')
    ax2.grid(True, which='both', linestyle='--', alpha=0.5)
    ax2.legend()
    
    plt.tight_layout()
    fig_path = FIGURES_DIR / f'ioh_comparison_f{p_id}.pdf'
    png_path = FIGURES_DIR / f'ioh_comparison_f{p_id}.png'
    plt.savefig(fig_path)
    plt.savefig(png_path)
    print(f'Saved figures for f{p_id} to {fig_path}')
    plt.show()